<a href="https://colab.research.google.com/github/georgegiannakidis/uhart-ms-prep/blob/main/Week1_Day2_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Week 1 — Day 2: Pandas Essentials
### UHart MS CS Pre-Arrival Prep · George · 7:00–8:30 PM
---
**Tonight's structure:**
- `7:00–7:05` → Recap NumPy (run the self-check below)
- `7:05–7:55` → Learn Pandas: DataFrames, filtering, groupby, cleaning
- `7:55–8:20` → 10 practice exercises on a real dataset
- `8:20–8:30` → 3 key takeaways + commit to GitHub

> 💡 **Watch first:** [Complete Python Pandas Tutorial 2024 — Keith Galli](https://www.youtube.com/watch?v=2uvysYbKdjM) (watch up to "Filtering" chapter ~1.5 hrs), then come back here.

---
## ⏮️ NumPy Recap (7:00–7:05)

In [ ]:
import numpy as np
# Quick warmup — should feel natural from yesterday
arr = np.array([[1,2,3],[4,5,6],[7,8,9]])
print('Shape:', arr.shape)
print('Row sums:', arr.sum(axis=1))
print('Normalized:', np.round((arr - arr.min()) / (arr.max() - arr.min()), 2))
print('✅ NumPy warmup done — moving to Pandas!')

---
## 📦 Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f'Pandas version: {pd.__version__}')
print('✅ Ready!')

---
## 1️⃣ Creating DataFrames

A **DataFrame** is Pandas' core structure — think of it as a spreadsheet in Python.
- Rows = observations (e.g. one student, one packet, one vulnerability)
- Columns = features (e.g. name, score, severity)

In [ ]:
# Create from a dictionary
data = {
    'System':   ['WebApp', 'Database', 'API', 'Auth', 'VPN'],
    'CVSS':     [9.8, 7.2, 4.5, 9.1, 6.8],
    'Severity': ['Critical', 'High', 'Medium', 'Critical', 'Medium'],
    'Patched':  [False, True, True, False, True]
}
df = pd.DataFrame(data)
print(df)
print()
print('Shape:', df.shape)     # (rows, cols)
print('Columns:', df.columns.tolist())
print('Dtypes:\n', df.dtypes)

In [ ]:
# Exploring a DataFrame — your daily workflow as a GRC analyst
print(df.head(3))       # first 3 rows
print()
print(df.tail(2))       # last 2 rows
print()
print(df.describe())    # stats for numeric columns
print()
print(df.info())        # column types & null counts

---
## 2️⃣ Accessing Data

Pandas has multiple ways to select data — know these cold.

In [ ]:
# Select a single column → returns a Series
cvss_col = df['CVSS']
print('CVSS column:\n', cvss_col)
print('Type:', type(cvss_col))

In [ ]:
# Select multiple columns → returns a DataFrame
subset = df[['System', 'CVSS', 'Severity']]
print(subset)

In [ ]:
# .iloc → select by INTEGER position (like NumPy)
print('Row 0:', df.iloc[0])          # first row
print()
print('Rows 1-2, cols 0-1:\n', df.iloc[1:3, 0:2])

In [ ]:
# .loc → select by LABEL (column name / index label)
print(df.loc[0, 'System'])           # single value
print()
print(df.loc[:, 'CVSS':'Severity'])  # slice of columns

---
## 3️⃣ Filtering Data

This is the most-used Pandas skill in GRC — filtering systems by risk, status, compliance.

In [ ]:
# Filter rows where CVSS >= 9.0 (Critical)
critical = df[df['CVSS'] >= 9.0]
print('Critical systems:\n', critical)

In [ ]:
# Multiple conditions — use & (and) | (or), always with parentheses!
high_unpatched = df[(df['CVSS'] >= 7.0) & (df['Patched'] == False)]
print('High/Critical AND unpatched:\n', high_unpatched)

In [ ]:
# Filter by string value
critical_sev = df[df['Severity'] == 'Critical']
print('Critical severity:\n', critical_sev)

# .isin() — filter by multiple values
hi_or_crit = df[df['Severity'].isin(['Critical', 'High'])]
print('\nHigh or Critical:\n', hi_or_crit)

---
## 4️⃣ Adding, Modifying & Renaming Columns

In [ ]:
# Add a new column
df['Risk Score'] = df['CVSS'] * 10   # scale to 0-100
df['Days to Patch'] = [30, 0, 0, 45, 0]  # 0 = already patched

# Conditional column with np.where (like IF() in Excel)
df['Action'] = np.where(df['Patched'], 'Monitor', 'REMEDIATE NOW')

print(df)

In [ ]:
# Rename columns
df_renamed = df.rename(columns={'CVSS': 'CVSS Score', 'Risk Score': 'Risk (0-100)'})
print(df_renamed.columns.tolist())

---
## 5️⃣ Handling Missing Data

Real-world data always has gaps — knowing how to handle NaN is critical.

In [ ]:
# Create a DataFrame with missing values
messy = pd.DataFrame({
    'System': ['A', 'B', 'C', 'D', 'E'],
    'CVSS':   [9.8, None, 4.5, None, 6.8],
    'Owner':  ['Alice', 'Bob', None, 'Dana', None]
})

print('Before cleaning:\n', messy)
print()
print('Null counts:\n', messy.isnull().sum())

In [ ]:
# Options for handling nulls:
print('Drop rows with any null:\n', messy.dropna())
print()
print('Fill CVSS nulls with mean:\n', messy['CVSS'].fillna(messy['CVSS'].mean()))
print()
print('Fill Owner nulls with Unknown:\n', messy['Owner'].fillna('Unknown'))

---
## 6️⃣ GroupBy & Aggregation

GroupBy = the most powerful Pandas feature. Split data → Apply function → Combine results.

In [ ]:
# Back to our vulnerability DataFrame
print('Average CVSS by Severity:')
print(df.groupby('Severity')['CVSS'].mean())

print()
print('Count of systems by Severity:')
print(df.groupby('Severity')['System'].count())

print()
print('Multiple aggregations at once:')
print(df.groupby('Severity')['CVSS'].agg(['mean', 'min', 'max', 'count']))

---
## 7️⃣ Sorting & Value Counts

In [ ]:
# Sort by CVSS score descending (highest risk first)
print('Sorted by risk (descending):\n', df.sort_values('CVSS', ascending=False))

print()
# value_counts — frequency of each category
print('Severity distribution:\n', df['Severity'].value_counts())
print()
print('Patched distribution:\n', df['Patched'].value_counts())

---
## 8️⃣ Loading Real Data (CSV)

In practice you'll always load data from files, not create DataFrames by hand.

In [ ]:
# Download the Titanic dataset — a classic ML practice dataset
import io, urllib.request

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
try:
    titanic = pd.read_csv(url)
    print('Titanic dataset loaded! Shape:', titanic.shape)
    print(titanic.head())
    print()
    print('Missing values:\n', titanic.isnull().sum())
except Exception as e:
    print(f'Could not load from URL. Using sample data instead.')
    titanic = pd.DataFrame({'PassengerId':[1,2,3],'Survived':[0,1,1],
                           'Pclass':[3,1,3],'Age':[22,38,26],'Fare':[7.25,71.28,7.92]})
    print(titanic)

---
## 🏋️ Practice Exercises (7:55–8:20 PM)

All exercises use the Titanic dataset. Try without hints first!

In [ ]:
# Reload the vulnerability DataFrame for exercises 1-5
df = pd.DataFrame({
    'System':   ['WebApp', 'Database', 'API', 'Auth', 'VPN', 'Mail', 'DNS', 'Firewall'],
    'CVSS':     [9.8, 7.2, 4.5, 9.1, 6.8, 8.5, 9.3, 3.2],
    'Severity': ['Critical','High','Medium','Critical','Medium','High','Critical','Low'],
    'Patched':  [False, True, True, False, True, False, False, True],
    'Owner':    ['Alice','Bob','Alice','Carol','Bob','Carol','Alice', None]
})
print(df)

In [ ]:
# ── Exercise 1 ──────────────────────────────────────────────────
# Filter all UNPATCHED systems with CVSS > 8.0
# Then sort by CVSS descending

# YOUR CODE HERE

In [ ]:
# ── Exercise 2 ──────────────────────────────────────────────────
# Add a column called 'Priority' using these rules:
# CVSS >= 9.0 → 'P1 - Critical'
# CVSS >= 7.0 → 'P2 - High'
# else       → 'P3 - Standard'
# Hint: use np.select() or nested np.where()

# YOUR CODE HERE

In [ ]:
# ── Exercise 3 ──────────────────────────────────────────────────
# Group by Owner and compute:
# - Number of systems each person owns
# - Average CVSS score per owner
# Hint: groupby + agg({'col1': 'count', 'col2': 'mean'})

# YOUR CODE HERE

In [ ]:
# ── Exercise 4 ──────────────────────────────────────────────────
# Fill the missing Owner value with 'Unassigned'
# Then count how many systems each owner has (including Unassigned)

# YOUR CODE HERE

In [ ]:
# ── Exercise 5 ──────────────────────────────────────────────────
# Create a summary DataFrame with ONE ROW per Severity level containing:
# - Count of systems
# - Number unpatched
# - Average CVSS
# Hint: multiple groupby + rename + merge, OR agg with a dict

# YOUR CODE HERE

In [ ]:
# ── Exercises 6-10 use Titanic dataset ──────────────────────────
# (run this first to make sure titanic is loaded)
try:
    titanic.shape
except NameError:
    titanic = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
print('Columns:', titanic.columns.tolist())

In [ ]:
# ── Exercise 6 ──────────────────────────────────────────────────
# What was the survival rate (%) for each passenger class (Pclass)?
# Hint: groupby('Pclass')['Survived'].mean() * 100

# YOUR CODE HERE

In [ ]:
# ── Exercise 7 ──────────────────────────────────────────────────
# How many passengers were in each class AND survived?
# Show as a pivot table with Pclass as rows, Survived as columns
# Hint: pd.crosstab()

# YOUR CODE HERE

In [ ]:
# ── Exercise 8 ──────────────────────────────────────────────────
# Fill missing Age values with the MEDIAN age
# Then show the before/after null count for the Age column

# YOUR CODE HERE

In [ ]:
# ── Exercise 9 ──────────────────────────────────────────────────
# Find the top 5 most expensive fares in the dataset
# Show only: Name, Pclass, Fare, Survived

# YOUR CODE HERE

In [ ]:
# ── Exercise 10 — GRC Bonus ──────────────────────────────────────
# Create a 'Risk Category' column on the vuln DataFrame:
# - Unpatched + CVSS >= 9 → 'IMMEDIATE ACTION'
# - Unpatched + CVSS >= 7 → 'URGENT'
# - Patched OR CVSS < 7   → 'MONITORED'
# Then use value_counts() to show the risk distribution
# And print total number of systems needing immediate or urgent action

df = pd.DataFrame({
    'System':  ['WebApp','Database','API','Auth','VPN','Mail','DNS','Firewall'],
    'CVSS':    [9.8, 7.2, 4.5, 9.1, 6.8, 8.5, 9.3, 3.2],
    'Patched': [False, True, True, False, True, False, False, True]
})
# YOUR CODE HERE

---
## 📝 Session Log (8:20–8:30 PM)

### 3 things I learned tonight:
1. *(write here)*
2. *(write here)*
3. *(write here)*

### GRC connection:
*(e.g. groupby = grouping vulnerabilities by owner for remediation tracking — this is exactly what a GRC analyst does in a risk register)*

### Tomorrow:
**Day 3 — Matplotlib & Scikit-learn** · Watch Corey Schafer Matplotlib Parts 1, 2, 6, 7 first

In [ ]:
# Answer Key — only after attempting!
import numpy as np, pandas as pd

df = pd.DataFrame({
    'System':  ['WebApp','Database','API','Auth','VPN','Mail','DNS','Firewall'],
    'CVSS':    [9.8, 7.2, 4.5, 9.1, 6.8, 8.5, 9.3, 3.2],
    'Severity':['Critical','High','Medium','Critical','Medium','High','Critical','Low'],
    'Patched': [False, True, True, False, True, False, False, True],
    'Owner':   ['Alice','Bob','Alice','Carol','Bob','Carol','Alice', None]
})

# Ex 1
print('Ex1:\n', df[(~df['Patched']) & (df['CVSS'] > 8.0)].sort_values('CVSS', ascending=False))

# Ex 2
conditions = [df['CVSS'] >= 9.0, df['CVSS'] >= 7.0]
choices    = ['P1 - Critical', 'P2 - High']
df['Priority'] = np.select(conditions, choices, default='P3 - Standard')
print('Ex2 Priority column:\n', df[['System','CVSS','Priority']])

# Ex 3
print('Ex3:\n', df.groupby('Owner').agg(systems=('System','count'), avg_cvss=('CVSS','mean')).round(1))

# Ex 4
df['Owner'] = df['Owner'].fillna('Unassigned')
print('Ex4:\n', df['Owner'].value_counts())

# Ex 10 (GRC)
df2 = pd.DataFrame({'System':['WebApp','Database','API','Auth','VPN','Mail','DNS','Firewall'],
                    'CVSS':[9.8,7.2,4.5,9.1,6.8,8.5,9.3,3.2],
                    'Patched':[False,True,True,False,True,False,False,True]})
c2 = [(~df2['Patched']) & (df2['CVSS'] >= 9), (~df2['Patched']) & (df2['CVSS'] >= 7)]
ch2 = ['IMMEDIATE ACTION', 'URGENT']
df2['Risk Category'] = np.select(c2, ch2, default='MONITORED')
print('Ex10:\n', df2['Risk Category'].value_counts())
urgent_count = (df2['Risk Category'].isin(['IMMEDIATE ACTION','URGENT'])).sum()
print('Systems needing action:', urgent_count)